Load Dataset

In [12]:
import pandas as pd
import numpy as np

# Load the Blood-Brain Barrier dataset
df = pd.read_csv('BloodBrain.csv')

# Dynamic target matching: standard column names for this dataset
possible_targets = ['logBB', 'log_ratio', 'log_brain_blood', 'class', 'logbb']
target_column = None

for col in possible_targets:
    if col in df.columns:
        target_column = col
        break

# Fallback: if none of the above match, default to the very last column in the file
if target_column is None:
    target_column = df.columns[-1]

print(f"Identified target column: '{target_column}'")

# Separate the target variable from the molecular features
X = df.drop(columns=[target_column])
y = df[target_column]

print(f"Dataset loaded successfully: {X.shape[0]} compounds, {X.shape[1]} molecular features.")

Identified target column: 'logBBB'
Dataset loaded successfully: 208 compounds, 134 molecular features.


Split the dataset

In [13]:
from sklearn.model_selection import train_test_split

# Split into 75% training and 25% test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

Training set: 156 samples | Test set: 52 samples


Select a learning method

In [14]:
from sklearn.ensemble import RandomForestRegressor

# Select the RandomForestRegressor for the regression task
rf_reg = RandomForestRegressor(random_state=42)

Define a tuning grid

In [15]:
from sklearn.model_selection import GridSearchCV

# 'max_features' maps directly to 'mtry' (number of features considered at each split)
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 'log2', 0.3, 0.5]
}

Perform 10-fold cross-validation

In [16]:
# Set up the GridSearch with 10-fold cross-validation evaluating Negative Mean Squared Error
grid_search = GridSearchCV(
    estimator=rf_reg, 
    param_grid=param_grid, 
    cv=10, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

# Run hyperparameter optimization on the training data
grid_search.fit(X_train, y_train)

# Extract and isolate the best trained model
best_model = grid_search.best_estimator_

c:\Users\Emilia\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Analyze performance values

In [17]:
# Convert negative MSE score from GridSearch back to positive RMSE
cv_mse = -grid_search.best_score_
cv_rmse = np.sqrt(cv_mse)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"10-Fold CV Root Mean Squared Error (RMSE): {cv_rmse:.4f}\n")

# Extract and rank molecular feature importances
importances = best_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Molecular Descriptor': X.columns, 
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features:")
print(feat_imp_df.head(10).to_string(index=False))

Best Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'n_estimators': 50}
10-Fold CV Root Mean Squared Error (RMSE): 0.5130

Top 10 Most Important Features:
Molecular Descriptor  Importance
                tpsa    0.038566
most_positive_charge    0.036664
                tcnp    0.036174
            psa_npsa    0.033151
               mlogp    0.032903
          polar_area    0.028247
              tpsa.1    0.027705
               fnsa3    0.024424
             vsa_acc    0.023184
              adistm    0.022744


Apply the final model to the test set

In [18]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions on unseen test compounds
y_pred = best_model.predict(X_test)

# Calculate regression performance metrics
test_mse = mean_squared_error(y_test, y_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_pred)

print(f"Test Set Mean Squared Error (MSE): {test_mse:.4f}")
print(f"Test Set Root Mean Squared Error (RMSE): {test_rmse:.4f}")
print(f"Test Set Coefficient of Determination (R² Score): {test_r2:.4f}")

Test Set Mean Squared Error (MSE): 0.2650
Test Set Root Mean Squared Error (RMSE): 0.5147
Test Set Coefficient of Determination (R² Score): 0.3662
